# Notebook 03: Feature Engineering

Convert raw data into numerical vectors that ML models can process.


## 1: Load Cleaned Data

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

events = pd.read_csv('../data/events_clean.csv')
users = pd.read_csv('../data/users_clean.csv')
categories = pd.read_csv('../data/categories_clean.csv')
registrations = pd.read_csv('../data/registrations_clean.csv')
attendance = pd.read_csv('../data/attendance_clean.csv')
feedback = pd.read_csv('../data/feedback_clean.csv')
user_interests = pd.read_csv('../data/user_interests_clean.csv')
interests = pd.read_csv('../data/interests_clean.csv')

print(f'Events: {len(events)}, Users: {len(users)}')
print(f'Registrations: {len(registrations)}, Feedback: {len(feedback)}')

## 2: Build Event Feature Vectors

Each event becomes a numerical vector combining:
- **Category** -> one-hot encoded (6 dims)
- **Description** -> TF-IDF (100 dims)
- **Popularity** -> normalized registration count (1 dim)
- **Avg rating** -> normalized (1 dim)

Total: ~108 dimensions per event

In [ ]:
from app.features.event_features import EventFeatureBuilder

event_builder = EventFeatureBuilder(max_tfidf_features=100)
event_features, event_id_to_idx, idx_to_event_id = event_builder.fit_transform(
    events, registrations, feedback, categories
)

print(f'Event feature matrix shape: {event_features.shape}')
print(f'  = {len(events)} events x {event_features.shape[1]} features')
print(f'  Sparsity: {1 - (event_features.nnz / np.prod(event_features.shape)):.2%}')

In [ ]:
# Inspect the feature names
feature_names = event_builder.get_feature_names()
print(f'Total features: {len(feature_names)}')
print(f'\nFirst 10 (categories): {feature_names[:10]}')
print(f'\nSample TF-IDF features: {feature_names[6:16]}')
print(f'\nLast 2 (numeric): {feature_names[-2:]}')

In [ ]:
# Visualize: top TF-IDF words across all events
tfidf_names = event_builder.tfidf.get_feature_names_out()
tfidf_matrix = event_builder.tfidf.transform(events['description_clean'].fillna(''))
avg_scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

top_n = 20
top_indices = avg_scores.argsort()[-top_n:][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    [tfidf_names[i] for i in top_indices][::-1],
    [avg_scores[i] for i in top_indices][::-1],
    color='steelblue'
)
ax.set_title('Top 20 TF-IDF Words Across All Events', fontweight='bold')
ax.set_xlabel('Average TF-IDF Score')
plt.tight_layout()
plt.savefig('../data/top_tfidf_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 3: Build User Profile Vectors

A user profile = weighted average of event features they interacted with.

Signal weights:
- Interest match: 1.0
- Registration: 2.0
- Attendance: 3.0
- Rating 4-5: 4.0
- Rating 1-2: -1.0

In [ ]:
from app.features.user_features import UserFeatureBuilder

user_builder = UserFeatureBuilder(event_features, event_id_to_idx)
user_profiles, user_id_to_idx, idx_to_user_id, user_interaction_counts = user_builder.build_all_profiles(
    users, registrations, attendance, feedback, user_interests, events
)

n_zero = sum(1 for p in user_profiles.values() if np.allclose(p, 0))
print(f'User profiles built: {len(user_profiles)}')
print(f'Zero profiles (cold-start): {n_zero}')
print(f'Profile vector dimensions: {list(user_profiles.values())[0].shape}')

In [ ]:
# Interaction count distribution
counts = list(user_interaction_counts.values())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(counts, bins=20, color='coral', edgecolor='white')
ax.axvline(3, color='red', linestyle='--', label='Cold-start threshold (3)')
ax.set_title('User Interaction Counts', fontweight='bold')
ax.set_xlabel('Number of Registrations')
ax.set_ylabel('Number of Users')
ax.legend()
plt.tight_layout()
plt.savefig('../data/user_interaction_counts.png', dpi=150, bbox_inches='tight')
plt.show()

cold_start = sum(1 for c in counts if c < 3)
print(f'Cold-start users (<3 interactions): {cold_start}/{len(counts)}')

## 4: Build Interaction Matrix (for SVD)

A matrix where rows=users, columns=events, values=interaction scores.

Scoring: registration=1.0, +attendance=+2.0, +rating offset from 3

In [ ]:
from app.features.user_features import build_interaction_matrix

interaction_matrix, im_user_map, im_event_map = build_interaction_matrix(
    users, events, registrations, attendance, feedback
)

n_nonzero = np.count_nonzero(interaction_matrix)
total = interaction_matrix.size
sparsity = 1 - (n_nonzero / total)

print(f'Interaction matrix shape: {interaction_matrix.shape}')
print(f'Non-zero entries: {n_nonzero}')
print(f'Sparsity: {sparsity:.2%}')
print(f'Score range: [{interaction_matrix.min():.1f}, {interaction_matrix.max():.1f}]')
print(f'Mean score (non-zero): {interaction_matrix[interaction_matrix > 0].mean():.2f}')

In [ ]:
# Visualize a 30x30 slice of the interaction matrix
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    interaction_matrix[:30, :30],
    cmap='YlOrRd', cbar_kws={'label': 'Interaction Score'},
    xticklabels=False, yticklabels=False, ax=ax
)
ax.set_title('Interaction Matrix (30x30 sample)', fontweight='bold')
ax.set_xlabel('Events')
ax.set_ylabel('Users')
plt.tight_layout()
plt.savefig('../data/interaction_matrix_scored.png', dpi=150, bbox_inches='tight')
plt.show()

## 5: Save Feature Artifacts

Save transformers and matrices so Phase 5 can load them directly.

In [ ]:
import joblib
import os

model_dir = '../trained_models'
os.makedirs(model_dir, exist_ok=True)

# Save transformers
joblib.dump(event_builder, f'{model_dir}/event_feature_builder.pkl')
joblib.dump(user_builder, f'{model_dir}/user_feature_builder.pkl')

# Save mappings
joblib.dump(event_id_to_idx, f'{model_dir}/event_id_to_idx.pkl')
joblib.dump(idx_to_event_id, f'{model_dir}/idx_to_event_id.pkl')
joblib.dump(user_id_to_idx, f'{model_dir}/user_id_to_idx.pkl')
joblib.dump(idx_to_user_id, f'{model_dir}/idx_to_user_id.pkl')
joblib.dump(user_interaction_counts, f'{model_dir}/user_interaction_counts.pkl')

# Save matrices
joblib.dump(event_features, f'{model_dir}/event_features.pkl')
joblib.dump(user_profiles, f'{model_dir}/user_profiles.pkl')
joblib.dump(interaction_matrix, f'{model_dir}/interaction_matrix.pkl')

print('Saved artifacts:')
for f in sorted(os.listdir(model_dir)):
    if f.endswith('.pkl'):
        size = os.path.getsize(f'{model_dir}/{f}') / 1024
        print(f'  {f:40s} {size:.1f} KB')

## 6: Sanity Check

Pick a user, show their profile vector, and see which events are most similar.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Pick a user with enough interactions
active_users = [(uid, cnt) for uid, cnt in user_interaction_counts.items() if cnt >= 5]
sample_uid, sample_cnt = active_users[0]

print(f'Sample user: {sample_uid}')
print(f'Interactions: {sample_cnt}')

# Get their profile
profile = user_profiles[sample_uid].reshape(1, -1)

# Compute similarity to all events
scores = cosine_similarity(profile, event_features).flatten()
top_5 = scores.argsort()[-5:][::-1]

print(f'\nTop 5 recommended events (by cosine similarity):')
for rank, idx in enumerate(top_5, 1):
    eid = idx_to_event_id[idx]
    title = events[events['id'] == eid]['title'].values[0]
    print(f'  {rank}. [{scores[idx]:.3f}] {title}')

# Show what they actually registered for
user_regs = registrations[registrations['user_id'] == sample_uid]['event_id']
print(f'\nActually registered for:')
for eid in user_regs.head(5):
    title = events[events['id'] == eid]['title'].values
    if len(title) > 0:
        print(f'  - {title[0]}')